<a href="https://colab.research.google.com/github/lawrennd/qig-code/blob/main/examples/gibbs_lock_hamiltonian_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gibbs-Lock and Hamiltonian Extraction — End-to-End Companion

This notebook is the end-to-end companion for the paper  
**"Gibbs-Lock and the Emergence of Hamiltonian Structure in the Inaccessible Game"**  
(implements CIP-000D).

It traces the full chain described in the paper:

1. **Setup** — construct a Gibbs-locked qutrit pair and inspect its spectral geometry  
2. **Loewner kernel and iso-marginal tangency** — classify perturbation modes  
3. **GENERIC decomposition and Hamiltonian extraction** — decompose the constrained flow and extract $H_\text{eff}$  
4. **$\mu_0$ inference** — infer the uniform decay rate from a synthetic trajectory  

**Dependencies:** `qig` package with CIP-000C (`GibbsLockedFrame`, `infer_mu0`).

## Setup

In [ ]:
# Auto-install QIG package if not available
try:
    import qig
except ImportError:
    print('Installing QIG package...')
    %pip install -q git+https://github.com/lawrennd/qig-code.git
    import qig


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import expm

import qig
from qig import GibbsLockedFrame, infer_mu0
from qig.exponential_family import QuantumExponentialFamily
from qig.generic import effective_hamiltonian_coefficients, effective_hamiltonian_operator
from qig.structure_constants import compute_structure_constants
from qig.core import generic_decomposition

print('qig package loaded')
print('GibbsLockedFrame:', GibbsLockedFrame)
print('infer_mu0:', infer_mu0)

---
## Section 1 — Setup: Near-Generalised-Bell Gibbs-Locked Qutrit Pair

We construct a Hamiltonian that is **diagonal in the generalised Bell basis**:
$$
|\Phi_{mn}\rangle = \frac{1}{\sqrt{3}}\sum_{k=0}^{2}\omega^{km}|k,\,(k{+}n)\bmod 3\rangle,
\qquad \omega = e^{2\pi i/3}.
$$
The spectrum of $H$ has the ground state $|\Phi_{00}\rangle$ at eigenvalue $0$, one
neighbouring level $|\Phi_{01}\rangle$ at eigenvalue $\delta$, and the remaining seven
Bell states at eigenvalue $1$.

**Key properties of this testbed:**

- Any mixture of Bell states has *exactly* maximally mixed marginals
  ($\rho_A = I/3$, $\rho_B = I/3$), so $\rho_0$ is **exactly LME** for all $\beta$.
- At large $\beta$ the Gibbs state $\rho_0 = e^{-\beta H}/Z$ is **near-Bell**: close
  to the pure maximally entangled state $|\Phi_{00}\rangle\langle\Phi_{00}|$.
- $[K_0,H]=\beta[H,H]=0$, so the **Gibbs-lock condition holds exactly**.
- $\delta$ controls the spectral asymmetry (how distinct one Bell level is);
  $\beta$ controls how close we are to the pure Bell boundary.


In [ ]:
d = 3
D = d * d
omega = np.exp(2j * np.pi / d)   # primitive cube root of unity

def bell_state_vec(m, n):
    """Generalised Bell state |Phi_{mn}> = (1/sqrt(d)) sum_k omega^{km} |k,(k+n)%d>."""
    psi = np.zeros(D, dtype=complex)
    for k in range(d):
        psi[k * d + (k + n) % d] += omega ** (k * m) / np.sqrt(d)
    return psi

# Bell basis matrix: column m*d+n is |Phi_{mn}>
U_bell = np.column_stack([bell_state_vec(m, n) for m in range(d) for n in range(d)])
assert np.allclose(U_bell.conj().T @ U_bell, np.eye(D)), 'Bell basis not orthonormal'

# Hamiltonian diagonal in Bell basis:
#   |Phi_{00}>  eigenvalue 0      (ground state, reference Bell state)
#   |Phi_{01}>  eigenvalue delta  (one slightly distinct level)
#   all others  eigenvalue 1
delta = 0.1   # spectral asymmetry
beta  = 5.0   # large beta -> near pure Bell ground state (near-LME)

eigs_H = np.ones(D)
eigs_H[0] = 0.0    # |Phi_{00}>
eigs_H[1] = delta  # |Phi_{01}>
H = (U_bell @ np.diag(eigs_H) @ U_bell.conj().T).real

I3 = np.eye(d, dtype=complex)  # used in later cells

frame = GibbsLockedFrame(H, beta=beta, dims=[d, d])

rho0 = frame.rho0
rho_A = np.trace(rho0.reshape(d, d, d, d), axis1=1, axis2=3)
rho_B = np.trace(rho0.reshape(d, d, d, d), axis1=0, axis2=2)

print(f'GibbsLockedFrame: D={frame.D}, beta={frame.beta}, dims={frame.dims}')
print(f'Gibbs-lock residual ||[K0,H]||_F = {frame.gibbs_lock_residual():.2e}')
print(f'rho0 eigenvalues (near-Bell): {np.sort(np.linalg.eigvalsh(rho0))[::-1].round(6)}')
print(f'||rho_A - I/3||_F = {np.linalg.norm(rho_A - np.eye(d)/d):.2e}  (exactly 0: Bell mixture)')
print(f'||rho_B - I/3||_F = {np.linalg.norm(rho_B - np.eye(d)/d):.2e}  (exactly 0: Bell mixture)')


In [ ]:
eps, gaps = frame.bohr_gaps()
print('Eigenvalues of H (Bohr frequencies):', eps.round(4))
print('Unique non-zero |Bohr gaps|:', np.unique(np.abs(gaps[np.abs(gaps) > 1e-10])).round(4))

---
## Section 2 — Loewner Kernel and Iso-Marginal Tangency

The **Loewner divided-difference kernel** maps modular-generator perturbations $\delta K$ to density-matrix perturbations:
$$
(\delta\rho)_{ij} = c(\lambda_i, \lambda_j)\,(\delta K)_{ij},
\qquad
c(\lambda_i, \lambda_j) = \frac{\lambda_i - \lambda_j}{\log\lambda_i - \log\lambda_j}.
$$

A perturbation is **iso-marginal** if $\operatorname{tr}_{\neq k}(J_{\rho_0}(\delta K)) = 0$ for every subsystem $k$.

In [ ]:
C, vals, vecs = frame.loewner_kernel()

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(C, cmap='viridis')
ax.set_title('Loewner kernel $C_{ij}$ in eigenbasis of $\\rho_0$')
ax.set_xlabel('$j$'); ax.set_ylabel('$i$')
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

print(f'Kernel range: [{C.min():.4f}, {C.max():.4f}]')
print(f'LME limit (1/D = 1/9 = {1/9:.4f}); kernel mean = {C.mean():.4f}')

In [ ]:
# ---------------------------------------------------------------------------
# Perturbation 1: matched-index — (|0><1|)_A ⊗ (|1><1|)_B
#
#   Tr_B(δK) = |0><1|_A · Tr_B(|1><1|_B) = |0><1|_A · 1  ≠ 0
#   Tr_A(δK) = Tr_A(|0><1|_A) · |1><1|_B = 0
#
#   Expected: NOT iso-marginal (changes ρ_A, leaves ρ_B intact).
# ---------------------------------------------------------------------------
delta_K_matched = np.zeros((9, 9), dtype=complex)
delta_K_matched[0*3+1, 1*3+1] = 1.0   # (i_A=0,j_B=1) row, (i_A=1,j_B=1) col
delta_K_matched[1*3+1, 0*3+1] = 1.0   # hermitian conjugate

delta_rho_matched = frame.loewner_map(delta_K_matched)
drho_A_matched = np.trace(delta_rho_matched.reshape(3, 3, 3, 3), axis1=1, axis2=3)
drho_B_matched = np.trace(delta_rho_matched.reshape(3, 3, 3, 3), axis1=0, axis2=2)

print('Perturbation 1: matched-index  (|0,1><1,1| + h.c.)')
print(f'  Tr_B(δρ) Frobenius norm : {np.linalg.norm(drho_A_matched):.4e}  (expect non-zero)')
print(f'  Tr_A(δρ) Frobenius norm : {np.linalg.norm(drho_B_matched):.4e}  (expect 0)')
print(f'  is_iso_marginal         : {frame.is_iso_marginal(delta_K_matched)}  (expect False)')
print()

# ---------------------------------------------------------------------------
# Perturbation 2: doubly off-diagonal — (|0><1|)_A ⊗ (|0><2|)_B
#
#   Tr_B(δK) = |0><1|_A · Tr_B(|0><2|_B) = |0><1|_A · 0  = 0
#   Tr_A(δK) = Tr_A(|0><1|_A) · |0><2|_B = 0
#
#   Expected: iso-marginal (neither marginal is affected).
#   This holds by tensor structure alone, independent of ρ_0.
# ---------------------------------------------------------------------------
delta_K_doubly = np.zeros((9, 9), dtype=complex)
delta_K_doubly[0*3+0, 1*3+2] = 1.0   # (i_A=0,j_B=0) row, (i_A=1,j_B=2) col
delta_K_doubly[1*3+2, 0*3+0] = 1.0   # hermitian conjugate

delta_rho_doubly = frame.loewner_map(delta_K_doubly)
drho_A_doubly = np.trace(delta_rho_doubly.reshape(3, 3, 3, 3), axis1=1, axis2=3)
drho_B_doubly = np.trace(delta_rho_doubly.reshape(3, 3, 3, 3), axis1=0, axis2=2)

print('Perturbation 2: doubly off-diagonal  (|0,0><1,2| + h.c.)')
print(f'  Tr_B(δρ) Frobenius norm : {np.linalg.norm(drho_A_doubly):.4e}  (expect 0)')
print(f'  Tr_A(δρ) Frobenius norm : {np.linalg.norm(drho_B_doubly):.4e}  (expect 0)')
print(f'  is_iso_marginal         : {frame.is_iso_marginal(delta_K_doubly)}  (expect True)')
print()

# ---------------------------------------------------------------------------
# Perturbation 3: local operator on site A — λ₁ ⊗ I
#
#   Tr_B(δK) = λ₁ · Tr_B(I_B) = λ₁ · d_B  ≠ 0
#
#   Expected: NOT iso-marginal (changes ρ_A; ρ_B unchanged by A-symmetry).
# ---------------------------------------------------------------------------
lam1 = np.array([[0, 1, 0], [1, 0, 0], [0, 0, 0]], dtype=complex)
delta_K_local = np.kron(lam1, I3)

delta_rho_local = frame.loewner_map(delta_K_local)
drho_A_local = np.trace(delta_rho_local.reshape(3, 3, 3, 3), axis1=1, axis2=3)
drho_B_local = np.trace(delta_rho_local.reshape(3, 3, 3, 3), axis1=0, axis2=2)

print('Perturbation 3: local site-A operator  (λ₁ ⊗ I)')
print(f'  Tr_B(δρ) Frobenius norm : {np.linalg.norm(drho_A_local):.4e}  (expect non-zero)')
print(f'  Tr_A(δρ) Frobenius norm : {np.linalg.norm(drho_B_local):.4e}  (expect 0 by symmetry)')
print(f'  is_iso_marginal         : {frame.is_iso_marginal(delta_K_local)}  (expect False)')


---
## Section 3 — GENERIC Decomposition and Hamiltonian Extraction

We instantiate a `QuantumExponentialFamily` for the qutrit pair and project the
Gibbs state $\rho_0$ from the `GibbsLockedFrame` onto its natural parameters
$\theta_0$ (the near-LME base point). We then perturb slightly from $\theta_0$
and decompose the constrained-flow Jacobian $M(\theta_0)$ into symmetric $S$ and
antisymmetric $A$ parts.

From $A$ we extract the effective Hamiltonian $H_\text{eff} = \sum_c \eta_c F_c$.

In [ ]:
# Build exponential family for qutrit pair
exp_fam = QuantumExponentialFamily(d=3, n_sites=2)
ops_list = exp_fam.operators          # list of matrices
ops = np.array(ops_list)              # shape (n_ops, 9, 9)
n_ops = len(ops_list)
print(f'Exponential family: d=3, n_sites=2, n_ops={n_ops}')

# Project the Gibbs state rho0 onto the exponential-family natural parameters.
# rho(theta) = exp(sum_c theta_c F_c) / Z  =>  sum_c theta_c F_c = log(rho0) + const*I
# Project the traceless part of log(rho0) onto each basis operator.
from scipy.linalg import logm as matrix_log
log_rho0 = matrix_log(frame.rho0)
log_rho0_tl = log_rho0 - np.trace(log_rho0) / 9 * np.eye(9)  # traceless part
theta_0 = np.array([
    np.real(np.trace(log_rho0_tl.conj().T @ ops[c]))
    / np.real(np.trace(ops[c].conj().T @ ops[c]))
    for c in range(n_ops)
])

# Verify reconstruction
rho0_check = exp_fam.rho_from_theta(theta_0)
print(f'||rho0 - rho_from_theta(theta_0)||_F = {np.linalg.norm(frame.rho0 - rho0_check):.2e}  (expect << 1)')

# Perturb slightly from the Gibbs-state base point (our near-LME reference)
rng = np.random.default_rng(0)
theta_star = theta_0 + rng.standard_normal(n_ops) * 0.05
rho_star = exp_fam.rho_from_theta(theta_star)
print(f'Perturbation ||theta_star - theta_0|| = {np.linalg.norm(theta_star - theta_0):.4f}')
print(f'rho_star trace: {np.trace(rho_star).real:.6f}')


In [ ]:
# GENERIC decomposition
M = exp_fam.jacobian(theta_star)
S, A = generic_decomposition(M)

print(f'Jacobian shape: {M.shape}')
print(f'||S||_F = {np.linalg.norm(S):.4f}  (dissipative)')
print(f'||A||_F = {np.linalg.norm(A):.4f}  (reversible)')

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, mat, title in zip(axes, [S, A], ['S (symmetric)', 'A (antisymmetric)']):
    im = ax.imshow(mat, cmap='RdBu_r')
    ax.set_title(title)
    plt.colorbar(im, ax=ax)
plt.suptitle('GENERIC decomposition at $\\theta^*$')
plt.tight_layout()
plt.show()

In [ ]:
# Extract effective Hamiltonian from antisymmetric sector
# ops_list is the original list; ops is the array form for structure constants
ops_list = exp_fam.operators
f_abc = compute_structure_constants(ops)  # ops is np.array(exp_fam.operators)
# effective_hamiltonian_coefficients returns (eta, diagnostics_dict)
eta, extraction_info = effective_hamiltonian_coefficients(A, theta_star, f_abc)
H_eff = effective_hamiltonian_operator(eta, ops_list)  # needs list of matrices

print(f'H_eff Hermitian: {np.allclose(H_eff, H_eff.conj().T, atol=1e-8)}')
print(f'H_eff traceless: {abs(np.trace(H_eff)) < 1e-8}')
print(f'H_eff Frobenius norm: {np.linalg.norm(H_eff):.4f}')

# Verification: compare antisymmetric flow with commutator bracket
# The parameter-space antisymmetric flow induces rho flow via Kubo-Mori derivatives
param_flow = A @ theta_star
drho_rev = sum(param_flow[a] * exp_fam.rho_derivative(theta_star, a) for a in range(n_ops))
drho_comm = -1j * (H_eff @ rho_star - rho_star @ H_eff)

norm_rev = np.linalg.norm(drho_rev, 'fro')
norm_comm = np.linalg.norm(drho_comm, 'fro')
print(f'\n||d_rho_rev||    = {norm_rev:.4f}')
print(f'||-i[H_eff,rho]|| = {norm_comm:.4f}')
print()
print('Note: these norms differ because the Kubo-Mori inner product != commutator.')
print('See docs/source/theory/hamiltonian_extraction.rst for the theoretical context.')

---
## Section 4 — $\mu_0$ Inference and Resolution Floor

The uniform decay rate $\mu_0$ governs off-diagonal coherence decay:
$$|(\delta\rho)_{ij}(t)| = |(\delta\rho)_{ij}(0)|\, e^{-\mu_0 t}.$$

We build a synthetic trajectory from `GibbsLockedFrame.linearised_flow` (exact analytical solution)
and recover $\mu_0$ via `infer_mu0`.  This links to **Experiment 5** of
`hamiltonian_emergence_experiments.ipynb`, which connects $\mu_0$ to the Fisher-information
resolution floor.

In [ ]:
mu0_true = 0.4
n_points = 60
t_max = 5.0
times = np.linspace(0.0, t_max, n_points)

# Build perturbation directly in the H eigenbasis (ensures visible magnitudes)
_, H_vecs = frame._eigh_H()
_, gaps_mat = frame.bohr_gaps()

rng = np.random.default_rng(42)
dK0_eig = (rng.standard_normal((9, 9)) + 1j * rng.standard_normal((9, 9))) * 0.05
dK0_eig = (dK0_eig + dK0_eig.conj().T) / 2
# Ensure all off-diagonal amplitudes exceed the tol=1e-3 threshold
for i in range(9):
    for j in range(9):
        if i != j and abs(dK0_eig[i, j]) < 0.02:
            dK0_eig[i, j] = 0.025 + 0j

# Analytical trajectory: element-wise exp decay + phase rotation
rho_traj = np.zeros((n_points, 9, 9), dtype=complex)
for ti, t in enumerate(times):
    phase = np.exp((1j * beta * gaps_mat - mu0_true) * t)
    dR_eig_t = dK0_eig * phase
    delta_rho = H_vecs @ dR_eig_t @ H_vecs.conj().T
    rho_traj[ti] = rho0 + delta_rho

print(f'Trajectory built: {n_points} steps, t in [0, {t_max}]')

In [ ]:
mu0_fit = infer_mu0(times, rho_traj, frame)
print(f'True  mu_0         = {mu0_true:.4f}')
print(f'Inferred mu_0      = {mu0_fit:.4f}')
print(f'Relative error     = {abs(mu0_fit - mu0_true)/mu0_true*100:.2f}%')

In [ ]:
# Plot: mean off-diagonal decay vs exponential fits
delta_rho_eig = np.array([
    H_vecs.conj().T @ (rho_traj[ti] - rho0) @ H_vecs
    for ti in range(n_points)
])

mags = []
for i in range(9):
    for j in range(9):
        if i != j:
            m = np.abs(delta_rho_eig[:, i, j])
            if m[0] > 1e-3:
                mags.append(m / m[0])

mean_decay = np.mean(mags, axis=0)

fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogy(times, mean_decay, 'b-', lw=1.5, alpha=0.7, label='Mean off-diagonal magnitude')
ax.semilogy(times, np.exp(-mu0_true * times), 'r--', lw=2,
            label=f'True $e^{{-\\mu_0 t}}$, $\\mu_0={mu0_true}$')
ax.semilogy(times, np.exp(-mu0_fit * times), 'g:', lw=2,
            label=f'Fitted $e^{{-\\hat\\mu_0 t}}$, $\\hat\\mu_0={mu0_fit:.3f}$')
ax.set_xlabel('Time $t$')
ax.set_ylabel('Normalised off-diagonal magnitude')
ax.set_title('Uniform off-diagonal decay: true vs inferred $\\mu_0$')
ax.legend()
plt.tight_layout()
plt.show()

---
## Summary

| Step | Object / function | Key result |
|------|-------------------|------------|
| 1 | `GibbsLockedFrame(H, beta, dims=[3,3])` | $\|[K_0, H]\|_F < 10^{-14}$ |
| 1 | `frame.bohr_gaps()` | Bohr frequencies from spec$(H)$ |
| 2 | `frame.loewner_kernel()` | Divided-difference kernel $C_{ij}$ |
| 2 | `frame.is_iso_marginal(\delta K)` | Mode classification |
| 3 | `effective_hamiltonian_operator` | $H_\text{eff}$ Hermitian + traceless |
| 4 | `infer_mu0(times, rho_traj, frame)` | $\hat\mu_0$ within 5% of true |

### Cross-references

- `examples/hamiltonian_emergence_experiments.ipynb` — structural claim validation (CIP-000B)  
- `docs/source/theory/hamiltonian_extraction.rst` — Gibbs-lock theory and extraction algorithm  
- `qig.gibbs_lock` module documentation — `GibbsLockedFrame`, `infer_mu0`  
- CIP-000C and CIP-000D